In [1]:
import os
import glob
import sys

import pandas as pd
import scipy.stats as ss
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns

import torch
from torch.utils.data import DataLoader
import pytorch_lightning as pl

In [2]:
from legnet_embedding import LegNetEmbedding
sys.path.append("../../predictor/model/")
import utrdata_cl as utrdata
from pl_regressor import RNARegressor

## Loading data

In [3]:
utr_type = "utr3"

In [4]:
if utr_type == "utr5":
    MODEL_PATH = "../../predictor/regression_multiple/saved_models/model-utr5-deltas-epoch=9-step=840.ckpt"
elif utr_type == "utr3":
    MODEL_PATH = "../../predictor/regression_multiple/saved_models/model-utr3-deltas-epoch=9-step=1330.ckpt"
else:
    raise ValueError

In [5]:
PATHS_FROM_GENALG = f'../../generator/genetic_alg/results/{utr_type}/*.csv'
PATHS_FROM_DIFFUSION = f'../../generator/diffusion/generate/results/*{utr_type[-1]}UTR.csv'
cell_types = ["c1", "c2", "c4", "c6", "c13", "c17"]

In [6]:
import re

paths_genalg = glob.glob(PATHS_FROM_GENALG)
read_dfs_ga = []
for file in paths_genalg:
    bn = os.path.basename(file)
    ct_pair = re.match(r"utr\d-seed=\d+-genes=\d+-(c\d+-c\d+).csv", bn).group(1)
    df_chunk = pd.read_csv(file, header=0, usecols=["num", "seq"])
    df_chunk["generation_type"] = "genetic_algorithm"
    df_chunk["ct_pair"] = ct_pair
    read_dfs_ga.append(df_chunk)
df_ga = pd.concat(read_dfs_ga, axis=0)

In [7]:
paths_diffusion = glob.glob(PATHS_FROM_DIFFUSION)
read_dfs_dif = []
for file in paths_diffusion:
    bn = os.path.basename(file)
    ct_pair = re.match(r"cell_(c\d+)_epoch_\d+_\dUTR.csv", bn).group(1)
    df_chunk = pd.read_csv(file, header=0, usecols=["seq", "mass_center"])
    df_chunk.rename({"mass_center": "query_exp"}, axis=1, inplace=True)
    df_chunk["generation_type"] = "diffusion"
    df_chunk["ct_pair"] = ct_pair
    read_dfs_dif.append(df_chunk)
df_dif = pd.concat(read_dfs_dif, axis=0)

In [8]:
df = pd.concat([df_ga, df_dif]).drop("num", axis=1)
df

,seq,generation_type,ct_pair,query_exp
0,TGGTGCTTAGATCGTTAATGTGCTATATAGTTAGATTAGATGGAAT...,genetic_algorithm,c17-c2,NaN
1,CGTACAAATGCCAGCATGCCCAAGCGCAAGTAGTTACTACCAAGGA...,genetic_algorithm,c17-c2,NaN
2,TAAGATTAGCTTAGCTTCAGTAGATAAGCTGGAGGTAGAATACGCC...,genetic_algorithm,c17-c2,NaN
3,CGTGTTCTATTAGGCCGTTTGTATTACAGTCCAGCCGTTACGGTAG...,genetic_algorithm,c17-c2,NaN
4,TGTGCAGTACAGGATAGGGCGCCGTAGTTTAATTGCTAGTCAGCAA...,genetic_algorithm,c17-c2,NaN
...,...,...,...,...
1023995,CAGGCCTTGCTCAGGCCTCTGCCACTAAGCCGACAGGGATGGGGAG...,diffusion,c4,3.129343
1023996,TTCAATCTGCCAAGAAAACAGCAGTGGAAATGCTAATTTAAACTTG...,diffusion,c4,2.892987
1023997,CCCCCTTCTCGTGCCATTTGGACTCACCTGCATTTTATTACTAACC...,diffusion,c4,1.967321
1023998,AAGCCATGCCCCTGGAGACTGTCCTCCAAGTGAGGTCTCGTGTGAG...,diffusion,c4,2.408447


In [9]:
df.drop_duplicates(subset=["seq", "generation_type", "ct_pair"], inplace=True)
df.reset_index(drop=True, inplace=True)
df

,seq,generation_type,ct_pair,query_exp
0,TGGTGCTTAGATCGTTAATGTGCTATATAGTTAGATTAGATGGAAT...,genetic_algorithm,c17-c2,NaN
1,CGTACAAATGCCAGCATGCCCAAGCGCAAGTAGTTACTACCAAGGA...,genetic_algorithm,c17-c2,NaN
2,TAAGATTAGCTTAGCTTCAGTAGATAAGCTGGAGGTAGAATACGCC...,genetic_algorithm,c17-c2,NaN
3,CGTGTTCTATTAGGCCGTTTGTATTACAGTCCAGCCGTTACGGTAG...,genetic_algorithm,c17-c2,NaN
4,TGTGCAGTACAGGATAGGGCGCCGTAGTTTAATTGCTAGTCAGCAA...,genetic_algorithm,c17-c2,NaN
...,...,...,...,...
8119853,CAGGCCTTGCTCAGGCCTCTGCCACTAAGCCGACAGGGATGGGGAG...,diffusion,c4,3.129343
8119854,TTCAATCTGCCAAGAAAACAGCAGTGGAAATGCTAATTTAAACTTG...,diffusion,c4,2.892987
8119855,CCCCCTTCTCGTGCCATTTGGACTCACCTGCATTTTATTACTAACC...,diffusion,c4,1.967321
8119856,AAGCCATGCCCCTGGAGACTGTCCTCCAAGTGAGGTCTCGTGTGAG...,diffusion,c4,2.408447


In [10]:
df["seq"].duplicated().value_counts()

seq
False    8119855
True           3
Name: count, dtype: int64

In [11]:
df.drop_duplicates(subset=["seq"], inplace=True)
df.reset_index(drop=True, inplace=True)
df

,seq,generation_type,ct_pair,query_exp
0,TGGTGCTTAGATCGTTAATGTGCTATATAGTTAGATTAGATGGAAT...,genetic_algorithm,c17-c2,NaN
1,CGTACAAATGCCAGCATGCCCAAGCGCAAGTAGTTACTACCAAGGA...,genetic_algorithm,c17-c2,NaN
2,TAAGATTAGCTTAGCTTCAGTAGATAAGCTGGAGGTAGAATACGCC...,genetic_algorithm,c17-c2,NaN
3,CGTGTTCTATTAGGCCGTTTGTATTACAGTCCAGCCGTTACGGTAG...,genetic_algorithm,c17-c2,NaN
4,TGTGCAGTACAGGATAGGGCGCCGTAGTTTAATTGCTAGTCAGCAA...,genetic_algorithm,c17-c2,NaN
...,...,...,...,...
8119850,CAGGCCTTGCTCAGGCCTCTGCCACTAAGCCGACAGGGATGGGGAG...,diffusion,c4,3.129343
8119851,TTCAATCTGCCAAGAAAACAGCAGTGGAAATGCTAATTTAAACTTG...,diffusion,c4,2.892987
8119852,CCCCCTTCTCGTGCCATTTGGACTCACCTGCATTTTATTACTAACC...,diffusion,c4,1.967321
8119853,AAGCCATGCCCCTGGAGACTGTCCTCCAAGTGAGGTCTCGTGTGAG...,diffusion,c4,2.408447


In [12]:
df["seq"].duplicated().value_counts()

seq
False    8119855
Name: count, dtype: int64

In [13]:
current_cols = df.columns
current_cols

Index(['seq', 'generation_type', 'ct_pair', 'query_exp'], dtype='object')

In [14]:
for ct in cell_types:
    df[ct] = 0
df = pd.melt(df, id_vars=current_cols, value_vars=cell_types, var_name="cell_type", value_name="value").drop("value", axis=1)

In [15]:
df.sort_values(by=["ct_pair", "seq", "cell_type"], inplace=True)
df.reset_index(drop=True, inplace=True)

In [16]:
df.head(10)

,seq,generation_type,ct_pair,query_exp,cell_type
0,AAAAAAAAGGGTGTCCGTTTCACTTATGATAAAAAGCGCTTTCCGG...,genetic_algorithm,c1-c13,NaN,c1
1,AAAAAAAAGGGTGTCCGTTTCACTTATGATAAAAAGCGCTTTCCGG...,genetic_algorithm,c1-c13,NaN,c13
2,AAAAAAAAGGGTGTCCGTTTCACTTATGATAAAAAGCGCTTTCCGG...,genetic_algorithm,c1-c13,NaN,c17
3,AAAAAAAAGGGTGTCCGTTTCACTTATGATAAAAAGCGCTTTCCGG...,genetic_algorithm,c1-c13,NaN,c2
4,AAAAAAAAGGGTGTCCGTTTCACTTATGATAAAAAGCGCTTTCCGG...,genetic_algorithm,c1-c13,NaN,c4
5,AAAAAAAAGGGTGTCCGTTTCACTTATGATAAAAAGCGCTTTCCGG...,genetic_algorithm,c1-c13,NaN,c6
6,AAAAAAACCGATTGCACTCCAAGGAGAGAAAGCACTTTATCTATAG...,genetic_algorithm,c1-c13,NaN,c1
7,AAAAAAACCGATTGCACTCCAAGGAGAGAAAGCACTTTATCTATAG...,genetic_algorithm,c1-c13,NaN,c13
8,AAAAAAACCGATTGCACTCCAAGGAGAGAAAGCACTTTATCTATAG...,genetic_algorithm,c1-c13,NaN,c17
9,AAAAAAACCGATTGCACTCCAAGGAGAGAAAGCACTTTATCTATAG...,genetic_algorithm,c1-c13,NaN,c2


In [18]:
ct_classes = df["cell_type"].unique()
num_classes = ct_classes.shape[0]
num_classes

6

In [19]:
batch_size = 1024

In [20]:
num_workers = 32

In [21]:
test_set = utrdata.UTRData(
    df=df,
    predict_cols=[],
    features=("sequence", "positional", "conditions"),
    construct_type=utr_type,
    augment=False,
    augment_test_time=False,
    augment_kws=dict(
        extend_left=0,
        extend_right=0,
        shift_left=0,
        shift_right=0,
        revcomp=False,
    ),
)

In [22]:
# Creating DataLoaders
dl_test = DataLoader(
    test_set,
    batch_size=batch_size,
    num_workers=num_workers,
    shuffle=False,
    drop_last=False
)

In [23]:
ckpt = torch.load(MODEL_PATH)

In [24]:
ckpt["hyper_parameters"]["model_class"] = LegNetEmbedding
ckpt["hyper_parameters"]["model_class"]

legnet_embedding.LegNetEmbedding

In [25]:
loaded_model = RNARegressor(**ckpt["hyper_parameters"])
loaded_model.load_state_dict(ckpt["state_dict"])

<All keys matched successfully>

In [26]:
progressbar_callback = pl.callbacks.TQDMProgressBar(refresh_rate=0.5)
trainer = pl.Trainer(
    callbacks=[progressbar_callback],
    logger=False,
    accelerator="gpu",
    devices=[1],
    deterministic=True,
)

prediction = trainer.predict(model=loaded_model, dataloaders=dl_test)

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
IPU available: False, using: 0 IPUs
HPU available: False, using: 0 HPUs
You are using a CUDA device ('NVIDIA GeForce RTX 3090') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/generated/torch.set_float32_matmul_precision.html#torch.set_float32_matmul_precision
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]


Predicting: 0it [00:00, ?it/s]

In [27]:
pred_tuples, _ = zip(*prediction)
embed, pred = zip(*pred_tuples)
embed = torch.concat(embed).numpy()
pred = torch.concat(pred).numpy()

In [28]:
seq_embedding = embed.reshape(-1, num_classes * embed.shape[-1], order="C")
seq_embedding

array([[0.11815199, 0.19145025, 0.05650521, ..., 0.14793622, 0.16633844,
        0.13583474],
       [0.14843343, 0.12539129, 0.01631776, ..., 0.11920269, 0.13243322,
        0.10633778],
       [0.19895266, 0.21860787, 0.01508135, ..., 0.22404872, 0.16506267,
        0.18244112],
       ...,
       [0.12369621, 0.15728155, 0.04658493, ..., 0.23046574, 0.15068589,
        0.17116494],
       [0.09538012, 0.12982565, 0.02404123, ..., 0.3274246 , 0.16526414,
        0.19959177],
       [0.12134919, 0.12886447, 0.05692219, ..., 0.2205248 , 0.15653141,
        0.18581603]], dtype=float32)

In [29]:
pred_mass_center = pred[:, 1].reshape(-1, num_classes)
pred_mass_center

array([[2.8991895, 2.3784676, 2.7532432, 2.7635887, 2.6972857, 2.7721648],
       [2.6058807, 2.0936615, 2.0158882, 2.248438 , 2.101338 , 2.2405052],
       [3.0083156, 2.4233727, 2.890439 , 2.789711 , 2.8102846, 2.7933888],
       ...,
       [2.511478 , 2.5183215, 1.9661787, 2.4888823, 2.022318 , 2.5041282],
       [2.6715944, 2.6578648, 1.9253696, 2.753757 , 1.9202994, 2.8651536],
       [2.5810554, 2.5794168, 2.711666 , 2.6532469, 2.5978734, 2.7261727]],
      dtype=float32)

In [30]:
result = pd.DataFrame(
    pred_mass_center,
    columns=pd.MultiIndex.from_product([["pred_mass_center"], cell_types]),
    index=df.iloc[::num_classes]["seq"]
)
result["pred_mass_center"].head()

,c1,c2,c4,c6,c13,c17
seq,,,,,,
AAAAAAAAGGGTGTCCGTTTCACTTATGATAAAAAGCGCTTTCCGGATTCAGCACACTTGGATTTATTTTTGTTACGTTATGTATACTATGGTCATTTTGCGTCGTTTCTGTAGGTCTTTGCTCATTTTCCAAGCACTTACGTTAGGCAGGTGTCTCTTCATTTAATGATCAAAGGGCCGTTACTATTGGTCTGTAACAAGTTTATCACACTATTTGGCGCTTGCGAGTATTGCGTATCT,2.899189,2.378468,2.753243,2.763589,2.697286,2.772165
AAAAAAACCGATTGCACTCCAAGGAGAGAAAGCACTTTATCTATAGGAGGCTGTTGCTCGAATCAGCACTTCGACATAACCTGCTATCCAATATTTATGGAATTGTGTTTGTCGCATCTAATACTTATTAATGATCGATCTTATTCCGATCATTATTTTATGTTCCTGGTCCTTCTAAGTTAGTAGTAGGGCAATGCCATTATAGGCTAGCATTCGTCTAGGATTAGTGCACGCTAACCC,2.605881,2.093662,2.015888,2.248438,2.101338,2.240505
AAAAAAATTATATTTTACTTGATACACCCAATACTCTGTGATAAAAATCTCTGAGAGAGCTGTAGTAGGTATTTGCTTAATCGTTTCTTTATCGTCCACTTACGATTGTTGTGTTGCATGGGCGCCATGCACTTAGCGGGGCTATTAGCATCTATTATAATGCCAACAAAATTCATTAAACATATAAAGCGAGTTTTGCTTCCTGTGCTGTGCTTGCTTGGTTTCTGGCCCACCTGGCAC,3.008316,2.423373,2.890439,2.789711,2.810285,2.793389
AAAAAACAGATTTTGTCTAAGCTCTCATGAGGACTTAATTCAAATTATGAAGTTGGCACTTACTACCGACAGTCGATGATCCAATTTACTATCGATTGTGATAATATGTCCCCAAGGGGCTCCACTCAATTGGTAAAGCATCTCGTATCGAGCAACTAAGTCTCATATGTAAAAGCTGTATGTATATCACATAGTAGAATTTTGATTAGAGAATTATTTGCAAGATTCGATTGTAAATAA,2.519634,2.306930,2.723998,2.546811,2.531938,2.543890
AAAAAACCATTTCCTGTGCACGGATCCCAGATGCGAGTGGCGCCGACACCTACAGGAAGTATATAGACCACTTTTGATATAATCCCCGCCACACAGTTGTATGGCAGAATTGAAACACCAACGTACCGGTGTACCGGCTATCTAGGTTCGCACTCACTTATATAGGGTCAGATGAACTAATTTAGCAACATACTTGCTTTCGGCCGTTGGAGGCCCTTATCAGGGACCGCTATCTTAACT,2.393653,2.510741,2.465920,2.488684,2.400708,2.466284


In [31]:
seq_embedding_df = pd.DataFrame(
    seq_embedding,
    columns=pd.MultiIndex.from_product([
        ["embedding"],
        [f"{ct}_{i:03d}" for ct in result["pred_mass_center"].columns for i in range(embed.shape[-1])]
    ]),
    index=result.index)
result = pd.concat([result, seq_embedding_df], axis=1)

In [32]:
result["embedding"]

,c1_000,c1_001,c1_002,c1_003,c1_004,c1_005,c1_006,c1_007,c1_008,c1_009,...,c17_022,c17_023,c17_024,c17_025,c17_026,c17_027,c17_028,c17_029,c17_030,c17_031
seq,,,,,,,,,,,,,,,,,,,,,
AAAAAAAAGGGTGTCCGTTTCACTTATGATAAAAAGCGCTTTCCGGATTCAGCACACTTGGATTTATTTTTGTTACGTTATGTATACTATGGTCATTTTGCGTCGTTTCTGTAGGTCTTTGCTCATTTTCCAAGCACTTACGTTAGGCAGGTGTCTCTTCATTTAATGATCAAAGGGCCGTTACTATTGGTCTGTAACAAGTTTATCACACTATTTGGCGCTTGCGAGTATTGCGTATCT,0.118152,0.191450,0.056505,0.148716,0.039827,0.005905,0.293289,0.239195,0.038324,0.054211,...,0.128658,0.110856,0.066502,0.121616,0.138130,0.134175,0.129934,0.147936,0.166338,0.135835
AAAAAAACCGATTGCACTCCAAGGAGAGAAAGCACTTTATCTATAGGAGGCTGTTGCTCGAATCAGCACTTCGACATAACCTGCTATCCAATATTTATGGAATTGTGTTTGTCGCATCTAATACTTATTAATGATCGATCTTATTCCGATCATTATTTTATGTTCCTGGTCCTTCTAAGTTAGTAGTAGGGCAATGCCATTATAGGCTAGCATTCGTCTAGGATTAGTGCACGCTAACCC,0.148433,0.125391,0.016318,0.123948,0.025161,0.008530,0.318144,0.154910,0.081376,0.041971,...,0.095915,0.084217,0.023081,0.064295,0.098794,0.106322,0.119159,0.119203,0.132433,0.106338
AAAAAAATTATATTTTACTTGATACACCCAATACTCTGTGATAAAAATCTCTGAGAGAGCTGTAGTAGGTATTTGCTTAATCGTTTCTTTATCGTCCACTTACGATTGTTGTGTTGCATGGGCGCCATGCACTTAGCGGGGCTATTAGCATCTATTATAATGCCAACAAAATTCATTAAACATATAAAGCGAGTTTTGCTTCCTGTGCTGTGCTTGCTTGGTTTCTGGCCCACCTGGCAC,0.198953,0.218608,0.015081,0.127041,0.000789,-0.033986,0.350073,0.175199,0.076673,0.062826,...,0.131385,0.095879,0.048343,0.129361,0.135185,0.175201,0.101318,0.224049,0.165063,0.182441
AAAAAACAGATTTTGTCTAAGCTCTCATGAGGACTTAATTCAAATTATGAAGTTGGCACTTACTACCGACAGTCGATGATCCAATTTACTATCGATTGTGATAATATGTCCCCAAGGGGCTCCACTCAATTGGTAAAGCATCTCGTATCGAGCAACTAAGTCTCATATGTAAAAGCTGTATGTATATCACATAGTAGAATTTTGATTAGAGAATTATTTGCAAGATTCGATTGTAAATAA,0.104462,0.124336,0.042089,0.091519,0.069728,0.141098,0.058125,0.103193,0.155782,0.128671,...,0.060930,0.042828,0.029590,0.095861,0.136168,0.151313,0.093905,0.177756,0.125524,0.159015
AAAAAACCATTTCCTGTGCACGGATCCCAGATGCGAGTGGCGCCGACACCTACAGGAAGTATATAGACCACTTTTGATATAATCCCCGCCACACAGTTGTATGGCAGAATTGAAACACCAACGTACCGGTGTACCGGCTATCTAGGTTCGCACTCACTTATATAGGGTCAGATGAACTAATTTAGCAACATACTTGCTTTCGGCCGTTGGAGGCCCTTATCAGGGACCGCTATCTTAACT,0.067919,0.110286,0.080103,0.099914,0.091392,0.123830,0.022989,0.110576,0.161323,0.141744,...,0.080969,0.081506,0.053720,0.108973,0.116858,0.146397,0.037558,0.159160,0.124345,0.159379
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
TTTTTTTGAAGCAATACGGTCAAAACTATTACCTATATTATTGTGTAGCGAATTTATTGTAAACACTCTCAGGCAAAGAGACTAGCCCTTAAATATCTTACGTATAGTTCCCCTCCCTTATCGGGAGCGGCAAATAAAACTACTCGTTGTACGGCTCAGTATAGTTGATAGTAATACCACCCTTGTCCCCCACACTATTATTAACATAAACACTACGTCAATTTGTATCATAAGTGTACG,0.199693,0.123627,0.020761,0.117425,0.025595,0.044105,0.242771,0.119185,0.129272,0.076094,...,0.040044,0.014255,0.021951,0.094693,0.147797,0.192743,0.117236,0.239706,0.116695,0.188239
TTTTTTTGCAGGAATGCGGTCAAAATTATTACTCATATTCTCTTTTCCTGTCAATACTGAAAACACTAATAGTCAGTCTGGTACTGTCATAAGTTTAGGGCTACTCGGAAGTAGACGTAATGGCAGGGTTCATCACTTTAATGACACTATGTATCTCATGGATGAAAATAGATACTAATTCCGCAAGTACATGTGATGCCCCAGTGTCATATCATTACACAATTGTCTCCGAAGGGGACG,0.146034,0.130932,0.093317,0.156666,0.088792,0.060989,0.142259,0.152242,0.093309,0.114895,...,0.062513,0.062079,0.058307,0.094917,0.128187,0.154603,0.044567,0.236939,0.109638,0.180850
TTTTTTTGCAGGAATGCGGTCTAAATCATTACTCATATTCGGCTATAGTATCCCACTTCTGACTAGACAATCTTTGCTACGAAATGGTTCCTCTTCAGAGCATCGCCGTTGCCCCTATCATTCCGCGCCAGCTACTACCCGACGGGAGGTAATTACACTATTGTCAAATCTTCTGCCTCGGTACGGGTCTATCTCGTTCTGGCGTATTAACACTACTTCAAGTTGTATCCGACGGGGACG,0.123696,0.157282,0.046585,0.135373,0.015611,0.037432,0.158944,0.141920,0.090670,0.075846,...,0.045063,0.048992,0.028674,0.079928,0.119894,0.185662,0.113191,0.230466,0.150686,0.171165


### Counting k-mers

In [33]:
sys.path.append("../../benchmark/kmers/")
from kmer_model import KmerLinearRegressor

In [34]:
kmer_dfs = []
for k in [1, 2, 3]:
    kmerreg = KmerLinearRegressor(kmer_length=k)
    kmer_df_k = kmerreg.count_kmers(result.index.values)
    kmer_df_k.columns = pd.MultiIndex.from_product([
        ["kmer_counts"],
        kmer_df_k.columns
    ])
    kmer_dfs.append(kmer_df_k)
    print(f"k={k} done")
result = pd.concat([result] + kmer_dfs, axis=1)

k=1 done
k=2 done
k=3 done


In [35]:
result["kmer_counts"]

,A,C,G,T,AA,AC,AG,AT,CA,CC,...,TCG,TCT,TGA,TGC,TGG,TGT,TTA,TTC,TTG,TTT
seq,,,,,,,,,,,,,,,,,,,,,
AAAAAAAAGGGTGTCCGTTTCACTTATGATAAAAAGCGCTTTCCGGATTCAGCACACTTGGATTTATTTTTGTTACGTTATGTATACTATGGTCATTTTGCGTCGTTTCTGTAGGTCTTTGCTCATTTTCCAAGCACTTACGTTAGGCAGGTGTCTCTTCATTTAATGATCAAAGGGCCGTTACTATTGGTCTGTAACAAGTTTATCACACTATTTGGCGCTTGCGAGTATTGCGTATCT,56,44,49,91,17,11,10,18,14,4,...,1,6,2,4,4,6,9,6,8,15
AAAAAAACCGATTGCACTCCAAGGAGAGAAAGCACTTTATCTATAGGAGGCTGTTGCTCGAATCAGCACTTCGACATAACCTGCTATCCAATATTTATGGAATTGTGTTTGTCGCATCTAATACTTATTAATGATCGATCTTATTCCGATCATTATTTTATGTTCCTGGTCCTTCTAAGTTAGTAGTAGGGCAATGCCATTATAGGCTAGCATTCGTCTAGGATTAGTGCACGCTAACCC,68,49,45,78,18,9,15,26,13,10,...,5,5,1,5,2,5,10,5,4,5
AAAAAAATTATATTTTACTTGATACACCCAATACTCTGTGATAAAAATCTCTGAGAGAGCTGTAGTAGGTATTTGCTTAATCGTTTCTTTATCGTCCACTTACGATTGTTGTGTTGCATGGGCGCCATGCACTTAGCGGGGCTATTAGCATCTATTATAATGCCAACAAAATTCATTAAACATATAAAGCGAGTTTTGCTTCCTGTGCTGTGCTTGCTTGGTTTCTGGCCCACCTGGCAC,63,48,46,83,21,11,9,22,13,9,...,2,6,3,8,4,7,9,4,8,8
AAAAAACAGATTTTGTCTAAGCTCTCATGAGGACTTAATTCAAATTATGAAGTTGGCACTTACTACCGACAGTCGATGATCCAATTTACTATCGATTGTGATAATATGTCCCCAAGGGGCTCCACTCAATTGGTAAAGCATCTCGTATCGAGCAACTAAGTCTCATATGTAAAAGCTGTATGTATATCACATAGTAGAATTTTGATTAGAGAATTATTTGCAAGATTCGATTGTAAATAA,82,40,42,76,27,10,15,29,15,6,...,5,4,5,1,2,7,6,2,7,6
AAAAAACCATTTCCTGTGCACGGATCCCAGATGCGAGTGGCGCCGACACCTACAGGAAGTATATAGACCACTTTTGATATAATCCCCGCCACACAGTTGTATGGCAGAATTGAAACACCAACGTACCGGTGTACCGGCTATCTAGGTTCGCACTCACTTATATAGGGTCAGATGAACTAATTTAGCAACATACTTGCTTTCGGCCGTTGGAGGCCCTTATCAGGGACCGCTATCTTAACT,67,62,51,60,15,21,13,18,18,18,...,2,2,3,3,3,3,4,3,5,5
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
TTTTTTTGAAGCAATACGGTCAAAACTATTACCTATATTATTGTGTAGCGAATTTATTGTAAACACTCTCAGGCAAAGAGACTAGCCCTTAAATATCTTACGTATAGTTCCCCTCCCTTATCGGGAGCGGCAAATAAAACTACTCGTTGTACGGCTCAGTATAGTTGATAGTAATACCACCCTTGTCCCCCACACTATTATTAACATAAACACTACGTCAATTTGTATCATAAGTGTACG,75,53,36,76,23,19,12,21,14,15,...,2,2,2,0,0,7,8,1,7,7
TTTTTTTGCAGGAATGCGGTCAAAATTATTACTCATATTCTCTTTTCCTGTCAATACTGAAAACACTAATAGTCAGTCTGGTACTGTCATAAGTTTAGGGCTACTCGGAAGTAGACGTAATGGCAGGGTTCATCACTTTAATGACACTATGTATCTCATGGATGAAAATAGATACTAATTCCGCAAGTACATGTGATGCCCCAGTGTCATATCATTACACAATTGTCTCCGAAGGGGACG,71,46,47,76,20,15,12,24,19,6,...,1,5,4,3,3,6,5,4,2,9
TTTTTTTGCAGGAATGCGGTCTAAATCATTACTCATATTCGGCTATAGTATCCCACTTCTGACTAGACAATCTTTGCTACGAAATGGTTCCTCTTCAGAGCATCGCCGTTGCCCCTATCATTCCGCGCCAGCTACTACCCGACGGGAGGTAATTACACTATTGTCAAATCTTCTGCCTCGGTACGGGTCTATCTCGTTCTGGCGTATTAACACTACTTCAAGTTGTATCCGACGGGGACG,53,64,48,75,11,16,8,18,13,13,...,4,9,1,5,2,2,3,8,5,6


### Adding final info

In [36]:
dfseq = df.set_index("seq").iloc[::num_classes]
result["ct_pair"] = dfseq["ct_pair"]
result["generation_type"] = dfseq["generation_type"]
result["query_exp"] = dfseq["query_exp"]

In [37]:
result

pred_mass_center            \
                                                                 c1        c2   
seq                                                                             
AAAAAAAAGGGTGTCCGTTTCACTTATGATAAAAAGCGCTTTCCGGA...         2.899189  2.378468   
AAAAAAACCGATTGCACTCCAAGGAGAGAAAGCACTTTATCTATAGG...         2.605881  2.093662   
AAAAAAATTATATTTTACTTGATACACCCAATACTCTGTGATAAAAA...         3.008316  2.423373   
AAAAAACAGATTTTGTCTAAGCTCTCATGAGGACTTAATTCAAATTA...         2.519634  2.306930   
AAAAAACCATTTCCTGTGCACGGATCCCAGATGCGAGTGGCGCCGAC...         2.393653  2.510741   
...                                                             ...       ...   
TTTTTTTGAAGCAATACGGTCAAAACTATTACCTATATTATTGTGTA...         2.568054  2.535183   
TTTTTTTGCAGGAATGCGGTCAAAATTATTACTCATATTCTCTTTTC...         2.535056  2.488157   
TTTTTTTGCAGGAATGCGGTCTAAATCATTACTCATATTCGGCTATA...         2.511478  2.518322   
TTTTTTTGCGACTCTCTGGTTATTTGCTCTACTACAAATCACTCAAG...         2.671594  2.657865   
TTTTTTTTGATACGTAATATTCTCGTGGCGTTATTTGCGTAATAAGT...         2.581055  2.579417   

                                                                        \
                                                          c4        c6   
seq                                                                      
AAAAAAAAGGGTGTCCGTTTCACTTATGATAAAAAGCGCTTTCCGGA...  2.753243  2.763589   
AAAAAAACCGATTGCACTCCAAGGAGAGAAAGCACTTTATCTATAGG...  2.015888  2.248438   
AAAAAAATTATATTTTACTTGATACACCCAATACTCTGTGATAAAAA...  2.890439  2.789711   
AAAAAACAGATTTTGTCTAAGCTCTCATGAGGACTTAATTCAAATTA...  2.723998  2.546811   
AAAAAACCATTTCCTGTGCACGGATCCCAGATGCGAGTGGCGCCGAC...  2.465920  2.488684   
...                                                      ...       ...   
TTTTTTTGAAGCAATACGGTCAAAACTATTACCTATATTATTGTGTA...  1.961185  2.581304   
TTTTTTTGCAGGAATGCGGTCAAAATTATTACTCATATTCTCTTTTC...  2.020684  2.474511   
TTTTTTTGCAGGAATGCGGTCTAAATCATTACTCATATTCGGCTATA...  1.966179  2.488882   
TTTTTTTGCGACTCTCTGGTTATTTGCTCTACTACAAATCACTCAAG...  1.925370  2.753757   
TTTTTTTTGATACGTAATATTCTCGTGGCGTTATTTGCGTAATAAGT...  2.711666  2.653247   

                                                                        \
                                                         c13       c17   
seq                                                                      
AAAAAAAAGGGTGTCCGTTTCACTTATGATAAAAAGCGCTTTCCGGA...  2.697286  2.772165   
AAAAAAACCGATTGCACTCCAAGGAGAGAAAGCACTTTATCTATAGG...  2.101338  2.240505   
AAAAAAATTATATTTTACTTGATACACCCAATACTCTGTGATAAAAA...  2.810285  2.793389   
AAAAAACAGATTTTGTCTAAGCTCTCATGAGGACTTAATTCAAATTA...  2.531938  2.543890   
AAAAAACCATTTCCTGTGCACGGATCCCAGATGCGAGTGGCGCCGAC...  2.400708  2.466284   
...                                                      ...       ...   
TTTTTTTGAAGCAATACGGTCAAAACTATTACCTATATTATTGTGTA...  1.934794  2.577770   
TTTTTTTGCAGGAATGCGGTCAAAATTATTACTCATATTCTCTTTTC...  1.977891  2.579379   
TTTTTTTGCAGGAATGCGGTCTAAATCATTACTCATATTCGGCTATA...  2.022318  2.504128   
TTTTTTTGCGACTCTCTGGTTATTTGCTCTACTACAAATCACTCAAG...  1.920299  2.865154   
TTTTTTTTGATACGTAATATTCTCGTGGCGTTATTTGCGTAATAAGT...  2.597873  2.726173   

                                                   embedding            \
                                                      c1_000    c1_001   
seq                                                                      
AAAAAAAAGGGTGTCCGTTTCACTTATGATAAAAAGCGCTTTCCGGA...  0.118152  0.191450   
AAAAAAACCGATTGCACTCCAAGGAGAGAAAGCACTTTATCTATAGG...  0.148433  0.125391   
AAAAAAATTATATTTTACTTGATACACCCAATACTCTGTGATAAAAA...  0.198953  0.218608   
AAAAAACAGATTTTGTCTAAGCTCTCATGAGGACTTAATTCAAATTA...  0.104462  0.124336   
AAAAAACCATTTCCTGTGCACGGATCCCAGATGCGAGTGGCGCCGAC...  0.067919  0.110286   
...                                                      ...       ...   
TTTTTTTGAAGCAATACGGTCAAAACTATTACCTATATTATTGTGTA...  0.199693  0.123627   
TTTTTTTGCAGGAATGCGGTCAAAATTATTACTCATATTCTCTTTTC...  0.146034  0.130932   
TTTTTTTGCAGGAATGCGGTCTAAATCA

In [38]:
result.to_parquet(f"embeddings_mapperless_{utr_type}_generated.parquet")

---